# Experiment: Controlled Berthoud Residual U-Net

Train the residual U-Net on the controlled Berthoud Pass mass/momentum dataset. This notebook expects ZIP artifacts created by `ml.residual_unet.prepare_colab_upload` after the WindNinja controlled run is complete.

## Drive Inputs

Upload these files to `MyDrive/windninja_ml/` before running this notebook:

- `residual_unet_code.zip`
- `controlled_berthoud_training_dataset.zip`

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = Path('/content').exists()
DRIVE_ROOT = Path('/content/drive/MyDrive/windninja_ml') if IN_COLAB else Path.cwd() / 'ml/residual_unet/outputs/colab_local'
REPO_DIR = Path('/content/mountain_windninja') if IN_COLAB else Path.cwd()
LOCAL_DATA_ROOT = Path('/content/data') if IN_COLAB else REPO_DIR / 'ml/residual_unet/data/processed'
DATASET_NAME = 'controlled_berthoud_training'
LOCAL_DATA = LOCAL_DATA_ROOT / DATASET_NAME
CODE_ZIP = DRIVE_ROOT / 'residual_unet_code.zip'
DATASET_ZIP = DRIVE_ROOT / f'{DATASET_NAME}_dataset.zip'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / DATASET_NAME
LOG_CSV = DRIVE_ROOT / 'logs' / f'{DATASET_NAME}_train_log.csv'
EVAL_OUT = DRIVE_ROOT / 'eval' / DATASET_NAME

REPO_DIR, LOCAL_DATA, CHECKPOINT_DIR

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV.parent.mkdir(parents=True, exist_ok=True)
EVAL_OUT.mkdir(parents=True, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## Unpack Code And Data

Training reads from local Colab disk. Checkpoints and metrics write back to Drive.

In [ ]:
if IN_COLAB:
    if not CODE_ZIP.exists():
        raise FileNotFoundError(f'Missing code ZIP: {CODE_ZIP}')
    if not DATASET_ZIP.exists():
        raise FileNotFoundError(f'Missing dataset ZIP: {DATASET_ZIP}')
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP) as archive:
        archive.extractall(REPO_DIR)
    if LOCAL_DATA.exists():
        shutil.rmtree(LOCAL_DATA)
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        archive.extractall(LOCAL_DATA_ROOT)

assert (REPO_DIR / 'ml/residual_unet/configs/controlled_berthoud_training.yaml').exists(), REPO_DIR
assert (LOCAL_DATA / 'manifest.csv').exists(), LOCAL_DATA
assert (LOCAL_DATA / 'normalization.json').exists(), LOCAL_DATA
print(f'Repo staged at: {REPO_DIR}')
print(f'Data staged at: {LOCAL_DATA}')

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'ml/residual_unet/requirements.txt')],
    check=True,
)

## Train

The configured run uses 80 epochs. Interruptions are recoverable from `latest.pt`.

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR)
cmd = [
    sys.executable,
    '-m',
    'ml.residual_unet.train',
    '--config',
    str(REPO_DIR / 'ml/residual_unet/configs/controlled_berthoud_training.yaml'),
    '--data',
    str(LOCAL_DATA),
    '--checkpoint-dir',
    str(CHECKPOINT_DIR),
    '--log-csv',
    str(LOG_CSV),
]
resume = CHECKPOINT_DIR / 'latest.pt'
if resume.exists():
    cmd += ['--resume', str(resume)]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

## Evaluate

The main pass evaluates held-out wind directions against the momentum solver target.

In [ ]:
cmd = [
    sys.executable,
    '-m',
    'ml.residual_unet.evaluate',
    '--checkpoint',
    str(CHECKPOINT_DIR / 'best.pt'),
    '--data',
    str(LOCAL_DATA),
    '--out',
    str(EVAL_OUT),
    '--split',
    'test',
]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
metrics = json.loads((EVAL_OUT / 'metrics.json').read_text())
metrics